# DRISHTI — Kaggle GPU quality runner
Recorded video + absolute GPS → offline georeferenced reconstruction. This notebook uses the supported, permissive-checkpoint baseline with a quality-focused profile and separate **30 decimal GB RAM / 30 decimal GB VRAM ceilings**.

Select a **GPU** accelerator in Kaggle Settings. The setup uses Python **3.11 or 3.12** for the project environment; if Kaggle's host kernel is newer, it provisions a compatible Python 3.12 environment with uv. Enable Internet for dependency installation; reconstruction uses attached local files only and sets offline model/PROJ flags. No model weights are downloaded by this notebook.

Attach three Kaggle Datasets: (1) the source ZIP made with `python scripts/package_kaggle.py`, (2) the Aukerman mission ZIP or your real video/GPS descriptor and optional logs, and (3) the verified `drishti-models.zip` package or complete local model directories plus their immutable/hash-verified `models.yaml`. Upload this `.ipynb` using Kaggle's **Import Notebook** and edit the configuration cell below.

The runner chooses **one actual GPU**; two T4s do not form one pooled 32 GB device. It uses lower measured stop budgets, a Torch allocator fraction, stage-isolated workers, process-tree telemetry and resumable bundles. Sampled peaks are lower bounds, not proof of an instantaneous hard limit. Quality, accuracy, memory fit and runtime still need a real mission run.

In [ ]:
from pathlib import Path

# Confirmed paths for your attached Kaggle Datasets. Kaggle mounted these
# Datasets as extracted directories under /kaggle/input/datasets, so the ZIP
# variables are None and the *_ROOT directories are used directly.
SOURCE_ZIP = None
SOURCE_ROOT = Path('/kaggle/input/datasets/suryanarayanhj/drishti-source/drishti')
MISSION_ZIP = None
MISSION_ROOT = Path('/kaggle/input/datasets/suryanarayanhj/aukerman-input')
DATASET_YAML = MISSION_ROOT / 'mission.yaml'
MODEL_ZIP = None
MODEL_ROOT = Path('/kaggle/input/datasets/suryanarayanhj/drishti-models')
MODEL_REGISTRY = MODEL_ROOT / 'models.yaml'
MODEL_CACHE = MODEL_ROOT
MODEL_PATHS = {
    'depth_anything_v2_small': str(MODEL_CACHE / 'depth_anything_v2_small'),
    'rtdetr': str(MODEL_CACHE / 'rtdetr'),  # Recommended; core motion fallback remains optional.
    # Add your registered semantic model here only if your descriptor enables it.
}

# For another Kaggle session, attach the extracted previous run directory here.
# The complete bundle includes its recorded inputs; resume preserves saved quality/cadence.
RESUME_BUNDLE = None  # e.g. Path('/kaggle/input/prior-run/kaggle-quality')
RUN_ID = 'kaggle-quality'
OUTPUT_ROOT = Path('/kaggle/working/drishti-runs')
PROJECT_ROOT = Path('/kaggle/working/drishti')
VENV_ROOT = Path('/tmp/drishti-kaggle-venv')  # Dependencies are not included in notebook outputs.
INSTALL_BLENDER = True  # Required for the prescribed out-of-process FBX exporter.
TORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu124'

# Decimal GB. Typed settings reject RAM or VRAM ceilings above 30 GB.
RESOURCE_SETTINGS = {
    'gpu_index': 0,
    'ram_limit_gb': 30.0,
    'vram_limit_gb': 30.0,
    'ram_reserve_gb': 3.0,
    'vram_reserve_gb': 1.0,
    'gpu_memory_fraction': 0.90,
    'cpu_threads': 8,
    'monitor_interval_s': 0.5,
    'min_system_available_gb': 1.0,
    'termination_grace_s': 5.0,
    'smi_timeout_s': 10.0,
    'report_recovery_timeout_s': 120.0,
}
SAMPLE_FRAME_BUDGET = 1800  # Approximate declared-duration budget across the FULL clip.
ARCHIVE_NAME = 'drishti-kaggle-quality.zip'
OVERRIDES = []  # Typed pipeline overrides, e.g. ['geo.resolution_m=0.15'].


## 1. Extract source and install the pinned runtime
This setup cell downloads Python/system dependencies only. It uses a separate virtual environment so the Kaggle kernel's packages are not replaced. Existing source, mission and model directories are reused only when their attached ZIP hashes match.

In [ ]:
import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import PurePosixPath

SUPPORTED_PYTHON = {(3, 11), (3, 12)}
TARGET_PYTHON = '3.12'
if sys.version_info[:2] not in SUPPORTED_PYTHON:
    print(f'Kaggle host Python {sys.version_info.major}.{sys.version_info.minor} is not supported by the project; provisioning Python {TARGET_PYTHON}.')

def resolve_attached_zip(configured, label):
    configured = Path(configured)
    if configured.is_file():
        return configured
    input_root = Path('/kaggle/input')
    candidates = sorted(input_root.rglob(configured.name)) if input_root.is_dir() else []
    if len(candidates) == 1:
        print(f'Configured {label} path was not present; using mounted input: {candidates[0]}')
        return candidates[0]
    mounted = sorted(str(path.relative_to(input_root)) for path in input_root.rglob('*')
                     if path.is_file()) if input_root.is_dir() else []
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple mounted {label} files named {configured.name}: {candidates}. Set the exact path in the configuration cell.')
    raise FileNotFoundError(f'Attach the {label} ZIP and set its exact path. Expected: {configured}. Mounted Kaggle inputs: {mounted[:100]}')

def resolve_attached_directory(configured, label, marker, predicate=lambda path: True):
    configured = Path(configured)
    if configured.is_dir() and (configured / marker).is_file() and predicate(configured):
        return configured
    input_root = Path('/kaggle/input')
    candidates = []
    if input_root.is_dir():
        for marker_path in input_root.rglob(marker):
            candidate = marker_path.parent
            if (candidate / marker).is_file() and predicate(candidate) and candidate not in candidates:
                candidates.append(candidate)
    candidates.sort()
    if len(candidates) == 1:
        print(f'Configured {label} directory was not present; using mounted input: {candidates[0]}')
        return candidates[0]
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple mounted {label} directories found: {candidates}. Set the exact path in the configuration cell.')
    raise FileNotFoundError(f'Attach the {label} Dataset and set its directory. Expected: {configured}')

source_zip = globals().get('SOURCE_ZIP')
source_root = (resolve_attached_directory(globals().get('SOURCE_ROOT', ''), 'source', 'pyproject.toml',
                                      lambda path: (path / 'src/drishti').is_dir())
               if source_zip is None else None)
if source_zip is not None:
    SOURCE_ZIP = resolve_attached_zip(source_zip, 'source')

# Extract the mission package atomically when the input Dataset contains a ZIP.
# The marker prevents silently reusing a different mission in /kaggle/working.
mission_zip = globals().get('MISSION_ZIP')
if mission_zip is None and 'MISSION_ROOT' in globals():
    MISSION_ROOT = resolve_attached_directory(MISSION_ROOT, 'mission', 'mission.yaml')
    if 'MISSION_ROOT' in globals():
        DATASET_YAML = MISSION_ROOT / 'mission.yaml'
if mission_zip is not None:
    mission_zip = resolve_attached_zip(mission_zip, 'mission')
    mission_hash = hashlib.sha256(mission_zip.read_bytes()).hexdigest()
    mission_marker = MISSION_ROOT / '.aukerman-input-sha256'
    MISSION_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if MISSION_ROOT.exists():
        if not mission_marker.is_file() or mission_marker.read_text().strip() != mission_hash:
            raise RuntimeError('MISSION_ROOT contains different/untracked input; choose a new MISSION_ROOT.')
        print('Reusing the matching extracted mission input.')
    else:
        with tempfile.TemporaryDirectory(prefix='drishti-mission-', dir=MISSION_ROOT.parent) as temporary:
            with zipfile.ZipFile(mission_zip) as archive:
                seen = set()
                for member in archive.infolist():
                    path = PurePosixPath(member.filename)
                    mode = member.external_attr >> 16
                    if (path.is_absolute() or '..' in path.parts or '\\' in member.filename
                            or not path.parts or stat.S_ISLNK(mode) or member.filename in seen):
                        raise ValueError(f'Invalid mission archive member: {member.filename}')
                    seen.add(member.filename)
                archive.extractall(temporary)
            if not (Path(temporary) / 'mission.yaml').is_file():
                raise ValueError('Mission ZIP is missing root mission.yaml.')
            os.replace(temporary, MISSION_ROOT)
        mission_marker.write_text(mission_hash + '\n')
    if 'MISSION_ROOT' in globals():
        DATASET_YAML = MISSION_ROOT / 'mission.yaml'

# Extract the verified model package when the input Dataset contains a ZIP.
model_zip = globals().get('MODEL_ZIP')
if model_zip is None and 'MODEL_ROOT' in globals():
    MODEL_ROOT = resolve_attached_directory(MODEL_ROOT, 'model', 'models.yaml',
                                      lambda path: (path / 'depth_anything_v2_small').is_dir())
if model_zip is not None:
    model_zip = resolve_attached_zip(model_zip, 'model')
    model_hash = hashlib.sha256(model_zip.read_bytes()).hexdigest()
    model_marker = MODEL_ROOT / '.drishti-models-sha256'
    MODEL_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if MODEL_ROOT.exists():
        if not model_marker.is_file() or model_marker.read_text().strip() != model_hash:
            raise RuntimeError('MODEL_ROOT contains different/untracked models; choose a new MODEL_ROOT.')
        print('Reusing the matching extracted model package.')
    else:
        with tempfile.TemporaryDirectory(prefix='drishti-models-', dir=MODEL_ROOT.parent) as temporary:
            with zipfile.ZipFile(model_zip) as archive:
                seen = set()
                for member in archive.infolist():
                    path = PurePosixPath(member.filename)
                    mode = member.external_attr >> 16
                    if (path.is_absolute() or '..' in path.parts or '\\' in member.filename
                            or not path.parts or stat.S_ISLNK(mode) or member.filename in seen):
                        raise ValueError(f'Invalid model archive member: {member.filename}')
                    seen.add(member.filename)
                archive.extractall(temporary)
            if not (Path(temporary) / 'models.yaml').is_file():
                raise ValueError('Model ZIP is missing root models.yaml.')
            os.replace(temporary, MODEL_ROOT)
        model_marker.write_text(model_hash + '\n')
    MODEL_REGISTRY = MODEL_ROOT / 'models.yaml'
    MODEL_CACHE = MODEL_ROOT
else:
    if 'MODEL_ROOT' in globals():
        MODEL_REGISTRY = MODEL_ROOT / 'models.yaml'
        MODEL_CACHE = MODEL_ROOT
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Select a GPU accelerator in Kaggle Settings before running setup.')

if source_zip is not None:
    hasher = hashlib.sha256()
    with SOURCE_ZIP.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            hasher.update(block)
    source_hash = hasher.hexdigest()
else:
    if not source_root.is_dir() or not (source_root / 'pyproject.toml').is_file():
        raise FileNotFoundError(f'Attach the extracted source Dataset and set SOURCE_ROOT: {source_root}')
    hasher = hashlib.sha256()
    for path in sorted(source_root.rglob('*')):
        if path.is_symlink():
            raise ValueError(f'Source Dataset contains an unsupported symlink: {path}')
        if path.is_file():
            hasher.update(path.relative_to(source_root).as_posix().encode())
            with path.open('rb') as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b''):
                    hasher.update(block)
    source_hash = hasher.hexdigest()
source_marker = PROJECT_ROOT / '.kaggle-source-sha256'
PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
if PROJECT_ROOT.exists():
    if not source_marker.is_file() or source_marker.read_text().strip() != source_hash:
        raise RuntimeError('PROJECT_ROOT contains different/untracked source; choose a new empty PROJECT_ROOT.')
    print('Reusing the matching extracted source checkout.')
else:
    if source_zip is not None:
        with tempfile.TemporaryDirectory(prefix='drishti-source-', dir=PROJECT_ROOT.parent) as temporary:
            with zipfile.ZipFile(SOURCE_ZIP) as archive:
                seen = set()
                for member in archive.infolist():
                    path = PurePosixPath(member.filename)
                    mode = member.external_attr >> 16
                    if (path.is_absolute() or '..' in path.parts or '\\' in member.filename
                            or not path.parts or path.parts[0] != 'drishti'
                            or stat.S_ISLNK(mode) or member.filename in seen):
                        raise ValueError(f'Invalid source archive member: {member.filename}')
                    seen.add(member.filename)
                archive.extractall(temporary)
            extracted = Path(temporary) / 'drishti'
            if not (extracted / 'pyproject.toml').is_file():
                raise ValueError('Source ZIP is missing drishti/pyproject.toml.')
            os.replace(extracted, PROJECT_ROOT)
    else:
        with tempfile.TemporaryDirectory(prefix='drishti-source-', dir=PROJECT_ROOT.parent) as temporary:
            extracted = Path(temporary) / 'drishti'
            shutil.copytree(source_root, extracted)
            os.replace(extracted, PROJECT_ROOT)
    source_marker.write_text(source_hash + '\n')

VENV_ROOT.parent.mkdir(parents=True, exist_ok=True)
venv_python = VENV_ROOT / 'bin/python'
if sys.version_info[:2] not in SUPPORTED_PYTHON and venv_python.is_file():
    existing_version = subprocess.run([str(venv_python), '-c', 'import sys; print(f\"{sys.version_info.major}.{sys.version_info.minor}\")'], capture_output=True, text=True, check=True).stdout.strip()
    if tuple(int(part) for part in existing_version.split('.')[:2]) not in SUPPORTED_PYTHON:
        shutil.rmtree(VENV_ROOT)
        venv_python = VENV_ROOT / 'bin/python'
    elif subprocess.run([str(venv_python), '-c', 'import importlib.util; print(importlib.util.find_spec(\"pip\") is not None)'], capture_output=True, text=True, check=True).stdout.strip() != 'True':
        print('Existing project environment has no pip; recreating it with a seeded pip installation.')
        shutil.rmtree(VENV_ROOT)
        venv_python = VENV_ROOT / 'bin/python'
if not venv_python.is_file():
    if sys.version_info[:2] in SUPPORTED_PYTHON:
        subprocess.run([sys.executable, '-m', 'venv', str(VENV_ROOT)], check=True)
    else:
        uv_prefix = VENV_ROOT.parent / 'uv-tools'
        uv_binary = shutil.which('uv')
        if uv_binary is None:
            uv_binary = str(uv_prefix / 'bin/uv')
            if not Path(uv_binary).is_file():
                subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir', '--prefix', str(uv_prefix), 'uv==0.6.14'], check=True)
        uv_env = {**os.environ, 'UV_PYTHON_INSTALL_DIR': str(VENV_ROOT.parent / 'python')}
        subprocess.run([str(uv_binary), 'python', 'install', TARGET_PYTHON], env=uv_env, check=True)
        subprocess.run([str(uv_binary), 'venv', '--seed', '--python', TARGET_PYTHON, str(VENV_ROOT)], env=uv_env, check=True)
PYTHON = VENV_ROOT / 'bin/python'
actual_python = subprocess.run([str(PYTHON), '-c', 'import sys; print(f\"{sys.version_info.major}.{sys.version_info.minor}\")'], capture_output=True, text=True, check=True).stdout.strip()
if tuple(int(part) for part in actual_python.split('.')[:2]) not in SUPPORTED_PYTHON:
    raise RuntimeError(f'Provisioned environment has unsupported Python {actual_python}; expected 3.11 or 3.12.')
print(f'Using project Python {actual_python}: {PYTHON}')
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--no-cache-dir',
                'torch==2.6.0', 'torchvision==0.21.0', '--index-url', TORCH_INDEX_URL], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--no-cache-dir', '-e',
                str(PROJECT_ROOT) + '[ingest,poses,neural,geo,server]'], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'check'], check=True)

if INSTALL_BLENDER and shutil.which('blender') is None:
    if shutil.which('apt-get') is not None and os.geteuid() == 0:
        install_env = {**os.environ, 'DEBIAN_FRONTEND': 'noninteractive'}
        subprocess.run(['apt-get', 'update'], env=install_env, check=True)
        subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends',
                        'blender', 'ffmpeg', 'libgl1', 'libgomp1'], env=install_env, check=True)
    else:
        print('Blender cannot be installed in this environment; FBX availability will be recorded by the exporter.')

subprocess.run(['nvidia-smi'], check=True)
probe = "import json, torch; assert torch.cuda.is_available(), 'An operational CUDA GPU is required'; print(json.dumps([{'index': i, 'name': torch.cuda.get_device_properties(i).name, 'vram_bytes': torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())], indent=2))"
subprocess.run([str(PYTHON), '-c', probe], check=True)
print('Setup finished. GPU detection does not imply that reconstruction or model inference ran.')


## 2. Validate inputs and write the runner settings
For the supplied Aukerman input, set `MISSION_ZIP` to the attached `aukerman-kaggle-input.zip`; setup extracts it and sets `DATASET_YAML` automatically. If you upload files individually, set `MISSION_ZIP = None` and point `MISSION_ROOT` at that input directory. For a new run, `mission.yaml` must point to the actual attached footage and GPS log. Relative paths resolve from the descriptor directory. Example:
```yaml
name: my-flight
video: flight.mp4
telemetry:
  format: dji_srt
  path: flight.srt
  altitude_datum: unknown
  time_origin_s: null
```
GPS requires absolute altitude. CSV/logged clocks require explicit origins; `unknown` is an explicit unvalidated altitude datum. Relative barometer height is optional and cannot replace GPS.

The model registry must contain actual immutable revisions, licenses and file hashes. `MODEL_PATHS` only rebinds those entries to Kaggle paths; it does not create checkpoints or relabel licensing. Required depth is the permissive **V2 Small**, not Base/Large. Put original optional-semantic/barometer settings in your dataset descriptor.

A run already present in `/kaggle/working` is resumed automatically. For a transferred run, set `RESUME_BUNDLE` to the extracted directory containing `manifest.json`. Saved sampling/profile settings are preserved on resume.

In [ ]:
if not MODEL_REGISTRY.is_file():
    raise FileNotFoundError(f'Attach the verified checkpoint registry: {MODEL_REGISTRY}')
resume = Path(RESUME_BUNDLE) if RESUME_BUNDLE is not None else None
effective_run_id = RUN_ID
if resume is not None:
    previous_manifest = json.loads((resume / 'manifest.json').read_text())
    effective_run_id = previous_manifest['run_id']
else:
    existing = OUTPUT_ROOT / RUN_ID
    if (existing / 'manifest.json').is_file():
        resume = existing

runner_settings = {
    **RESOURCE_SETTINGS,
    'output_root': str(OUTPUT_ROOT), 'run_id': effective_run_id,
    'registry': str(MODEL_REGISTRY), 'model_cache': str(MODEL_CACHE),
    'model_paths': MODEL_PATHS,
    'archive': True, 'archive_name': ARCHIVE_NAME, 'overrides': OVERRIDES,
}
if resume is not None:
    runner_settings['resume_bundle'] = str(resume)
    print(f'Resuming {resume}; saved profile and sampling cadence are preserved.')
else:
    if DATASET_YAML is None or not Path(DATASET_YAML).is_file():
        raise FileNotFoundError(f'Attach your real dataset descriptor: {DATASET_YAML}')
    runner_settings.update(dataset=str(DATASET_YAML),
                           profile=str(PROJECT_ROOT / 'configs/profiles/kaggle_quality.yaml'),
                           sample_frame_budget=SAMPLE_FRAME_BUDGET)

SETTINGS_FILE = OUTPUT_ROOT.parent / 'kaggle_run.yaml'
SETTINGS_FILE.parent.mkdir(parents=True, exist_ok=True)
SETTINGS_FILE.write_text(json.dumps(runner_settings, indent=2) + '\n')  # JSON is valid YAML.
validate = 'import sys; from drishti.runtime.kaggle import load_settings; print(load_settings(sys.argv[1]).model_dump_json(indent=2))'
subprocess.run([str(PYTHON), '-c', validate, str(SETTINGS_FILE)], cwd=PROJECT_ROOT, check=True)
print('Verified model hashes and measured GPU budgets are checked by the runner before stage work.')


## 3. Run the full reconstruction
The profile uses 1036-pixel Small depth inference, denser point fusion, larger matching/BA budgets, an 8192-pixel observed texture atlas and 0.20 m raster sampling. These are quality settings, **not accuracy claims**. Geometry remains float64/local-origin; published metric depth remains float32.

Every pipeline stage runs in a separate subprocess to release GPU/model state. Actual memory budgets are reduced below the configured ceilings and the selected GPU's capacity/free memory. A budget stop preserves real partial artifacts and attempts a bounded report. Quality is not silently lowered. Full-clip cadence is adjusted using declared duration metadata instead of truncating the flight tail.

Full-resolution images/depth products can use substantial disk space, particularly for 4K clips; ensure your Kaggle working/output allowance fits the mission. Stage logs are in the bundle's `logs/` directory.

In [ ]:
import signal
command = [str(PYTHON), str(PROJECT_ROOT / 'scripts/kaggle_run.py'),
           '--settings', str(SETTINGS_FILE)]
run_env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
process = subprocess.Popen(command, cwd=PROJECT_ROOT, env=run_env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    RUN_EXIT_CODE = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    print('Stopping the runner; allow worker cleanup, partial report and archive to finish.')
    RUN_EXIT_CODE = process.wait()
finally:
    process.stdout.close()
print(f'Runner exit code: {RUN_EXIT_CODE} (nonzero means partial/aborted; inspect outputs below).')


## 4. Inspect measured results and download
Use the Kaggle Output panel to download the source-independent reconstruction ZIP, HTML report and resource record. Keep the complete bundle and its georeference sidecars. For another session, attach/extract the bundle and set `RESUME_BUNDLE`; provide the same verified checkpoint inventory again.

`cloud_run.json` is the authoritative final orchestration record. The HTML/JSON report contains a labeled pre-recovery cloud snapshot; the ZIP contains a pre-creation snapshot and cannot recursively certify its own archive result. Sampled resource peaks may miss brief spikes. Absence of independent check data means absolute accuracy remains **UNVALIDATED**.

In [ ]:
from IPython.display import FileLink, display
BUNDLE = OUTPUT_ROOT / effective_run_id
cloud_path = BUNDLE / 'cloud_run.json'
if cloud_path.is_file():
    cloud = json.loads(cloud_path.read_text())
    limits = cloud.get('limits') or {}
    to_gb = lambda value: None if value is None else round(value / 1_000_000_000, 3)
    print(json.dumps({
        'status': cloud.get('status'), 'gpu': cloud.get('gpu'),
        'ram_stop_budget_gb': to_gb(limits.get('ram_bytes')),
        'vram_stop_budget_gb': to_gb(limits.get('vram_bytes')),
        'sampled_ram_peak_gb': to_gb(cloud.get('rss_peak_bytes')),
        'sampled_vram_peak_gb': to_gb(cloud.get('vram_peak_bytes')),
        'wall_seconds': cloud.get('orchestration_wall_seconds'),
        'sampling': cloud.get('sampling'), 'stop_reason': cloud.get('stop_reason'),
        'archive_status': cloud.get('archive_status'), 'archive_error': cloud.get('archive_error'),
    }, indent=2))
    archive = Path(cloud['archive_path']) if cloud.get('archive_path') else None
    for output in [archive, BUNDLE / 'report/report.html', cloud_path, BUNDLE / 'manifest.json']:
        if output is not None and output.is_file():
            display(FileLink(str(output)))
    verification = subprocess.run([str(PYTHON), '-m', 'drishti.cli', 'verify', str(BUNDLE)],
                                  cwd=PROJECT_ROOT, check=False)
    print(f'Bundle integrity exit code: {verification.returncode}; integrity is separate from accuracy.')
else:
    print('No bundle resource record exists. Read the input/checkpoint/setup error above.')
print('Download through Kaggle Output if notebook file links are not exposed by the UI.')
